# Подготовительная работа

Настраиваем пути, чтобы было видно импорты

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

Проверяем подключение к базе данных

In [3]:
from pandas import option_context
from psycopg2 import sql

from app.utils.db_connect import get_connection, query_to_pd
from config.db_config import DB_SCHEMA

In [3]:
version = query_to_pd("SELECT version() AS version")
print(version["version"].iloc[0])

PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


# EDA по таблицам и типам колонок

## Список таблиц

In [4]:
sql_tables = """
    SELECT tablename
    FROM pg_tables
    WHERE schemaname = %s;
"""
tables = query_to_pd(sql_tables, (DB_SCHEMA,))
display(tables)

,tablename
0,bureau
1,bureau_balance
2,pos_cash_balance
3,installments_payments
4,previous_application
5,credit_card_balance
6,application_train_test


In [5]:
sql_columns = """
    SELECT
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = %s AND table_name = %s
    ORDER BY ordinal_position;
"""
with get_connection() as conn:
    for table_name in tables["tablename"]:
        df_columns = query_to_pd(
            sql_columns, (DB_SCHEMA, table_name), conn=conn
        )
        print(f"Таблица {table_name}")
        with option_context(
            "display.max_rows", None, "display.max_columns", None
        ):
            display(df_columns)
        print()

Таблица bureau


,column_name,data_type,is_nullable,column_default
0,sk_id_curr,integer,YES,None
1,sk_id_bureau,integer,NO,None
2,credit_active,character varying,YES,None
3,credit_currency,character varying,YES,None
4,days_credit,smallint,YES,None
5,credit_day_overdue,smallint,YES,None
6,days_credit_enddate,real,YES,None
7,days_enddate_fact,real,YES,None
8,amt_credit_max_overdue,real,YES,None
9,cnt_credit_prolong,smallint,YES,None



Таблица bureau_balance


,column_name,data_type,is_nullable,column_default
0,sk_id_bureau,integer,YES,None
1,months_balance,smallint,YES,None
2,status,character varying,YES,None



Таблица pos_cash_balance


,column_name,data_type,is_nullable,column_default
0,sk_id_prev,integer,YES,None
1,sk_id_curr,integer,YES,None
2,months_balance,smallint,YES,None
3,cnt_instalment,real,YES,None
4,cnt_instalment_future,real,YES,None
5,name_contract_status,character varying,YES,None
6,sk_dpd,smallint,YES,None
7,sk_dpd_def,smallint,YES,None


... (4 outputs left out)


Таблица credit_card_balance


,column_name,data_type,is_nullable,column_default
0,sk_id_prev,integer,YES,None
1,sk_id_curr,integer,YES,None
2,months_balance,smallint,YES,None
3,amt_balance,real,YES,None
4,amt_credit_limit_actual,integer,YES,None
5,amt_drawings_atm_current,real,YES,None
6,amt_drawings_current,real,YES,None
7,amt_drawings_other_current,real,YES,None
8,amt_drawings_pos_current,real,YES,None
9,amt_inst_min_regularity,real,YES,None



Таблица application_train_test


,column_name,data_type,is_nullable,column_default
0,sk_id_curr,integer,NO,None
1,target,real,YES,None
2,name_contract_type,character varying,YES,None
3,code_gender,character varying,YES,None
4,flag_own_car,character varying,YES,None
5,flag_own_realty,character varying,YES,None
6,cnt_children,smallint,YES,None
7,amt_income_total,real,YES,None
8,amt_credit,real,YES,None
9,amt_annuity,real,YES,None


## Примеры данных

In [6]:
with get_connection() as conn:
    for table_name in tables["tablename"]:
        query = sql.SQL("SELECT * FROM {} LIMIT 3").format(
            sql.Identifier(table_name),
        )
        df = query_to_pd(query.as_string(conn), conn=conn)
        print(f"Таблица {table_name}")
        with option_context(
            "display.max_rows", None, "display.max_columns", None
        ):
            display(df)
        print()

Таблица bureau


,sk_id_curr,sk_id_bureau,credit_active,credit_currency,days_credit,credit_day_overdue,days_credit_enddate,days_enddate_fact,amt_credit_max_overdue,cnt_credit_prolong,amt_credit_sum,amt_credit_sum_debt,amt_credit_sum_limit,amt_credit_sum_overdue,credit_type,days_credit_update,amt_annuity
0,319805,6621080,Closed,currency 1,-1580,0,-1480.0,-1480.0,None,0,31455.0,0.0,None,0.0,Consumer credit,-1265,21055.5
1,319805,6621081,Closed,currency 1,-1527,0,-1316.0,-1082.0,None,0,98055.0,NaN,None,0.0,Consumer credit,-1079,65709.0
2,319805,6621082,Closed,currency 1,-668,0,-303.0,-606.0,None,0,225000.0,0.0,None,0.0,Consumer credit,-593,0.0



Таблица bureau_balance


,sk_id_bureau,months_balance,status
0,5502244,-75,C
1,5502244,-76,C
2,5502244,-77,C



Таблица pos_cash_balance


,sk_id_prev,sk_id_curr,months_balance,cnt_instalment,cnt_instalment_future,name_contract_status,sk_dpd,sk_dpd_def
0,2651600,225564,-80,6.0,6.0,Active,0,0
1,2178593,346239,-82,6.0,6.0,Active,0,0
2,2424838,182239,-78,8.0,8.0,Active,0,0


... (4 outputs left out)


Таблица credit_card_balance


,sk_id_prev,sk_id_curr,months_balance,amt_balance,amt_credit_limit_actual,amt_drawings_atm_current,amt_drawings_current,amt_drawings_other_current,amt_drawings_pos_current,amt_inst_min_regularity,amt_payment_current,amt_payment_total_current,amt_receivable_principal,amt_recivable,amt_total_receivable,cnt_drawings_atm_current,cnt_drawings_current,cnt_drawings_other_current,cnt_drawings_pos_current,cnt_instalment_mature_cum,name_contract_status,sk_dpd,sk_dpd_def
0,1311930,199563,-45,43953.797,45000,0.0,788.4,0.0,788.4,3150.0,3150.0,3150.0,42919.29,44716.320,44716.320,0.0,1,0.0,1.0,11.0,Active,0,0
1,1849318,165698,-53,43953.797,45000,0.0,788.4,0.0,788.4,3150.0,3150.0,3150.0,42919.29,44716.320,44716.320,0.0,1,0.0,1.0,11.0,Active,0,0
2,2205075,254726,-48,70012.890,90000,0.0,0.0,0.0,0.0,6300.0,6300.0,6300.0,69537.78,70925.805,70925.805,0.0,0,0.0,0.0,8.0,Active,0,0



Таблица application_train_test


,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_type_suite,name_income_type,name_education_type,name_family_status,name_housing_type,region_population_relative,days_birth,days_employed,days_registration,days_id_publish,own_car_age,flag_mobil,flag_emp_phone,flag_work_phone,flag_cont_mobile,flag_phone,flag_email,occupation_type,cnt_fam_members,region_rating_client,region_rating_client_w_city,weekday_appr_process_start,hour_appr_process_start,reg_region_not_live_region,reg_region_not_work_region,live_region_not_work_region,reg_city_not_live_city,reg_city_not_work_city,live_city_not_work_city,organization_type,ext_source_1,ext_source_2,ext_source_3,apartments_avg,basementarea_avg,years_beginexpluatation_avg,years_build_avg,commonarea_avg,elevators_avg,entrances_avg,floorsmax_avg,floorsmin_avg,landarea_avg,livingapartments_avg,livingarea_avg,nonlivingapartments_avg,nonlivingarea_avg,apartments_mode,basementarea_mode,years_beginexpluatation_mode,years_build_mode,commonarea_mode,elevators_mode,entrances_mode,floorsmax_mode,floorsmin_mode,landarea_mode,livingapartments_mode,livingarea_mode,nonlivingapartments_mode,nonlivingarea_mode,apartments_medi,basementarea_medi,years_beginexpluatation_medi,years_build_medi,commonarea_medi,elevators_medi,entrances_medi,floorsmax_medi,floorsmin_medi,landarea_medi,livingapartments_medi,livingarea_medi,nonlivingapartments_medi,nonlivingarea_medi,fondkapremont_mode,housetype_mode,totalarea_mode,wallsmaterial_mode,emergencystate_mode,obs_30_cnt_social_circle,def_30_cnt_social_circle,obs_60_cnt_social_circle,def_60_cnt_social_circle,days_last_phone_change,flag_document_2,flag_document_3,flag_document_4,flag_document_5,flag_document_6,flag_document_7,flag_document_8,flag_document_9,flag_document_10,flag_document_11,flag_document_12,flag_document_13,flag_document_14,flag_document_15,flag_document_16,flag_document_17,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,255040,0.0,Cash loans,F,Y,Y,0,144000.0,1007352.0,33421.5,765000.0,Unaccompanied,Commercial associate,Secondary / secondary special,Married,House / apartment,0.024610,-20658,-2276,-9765.0,-4174,8.0,1,1,0,1,1,0,NaN,2.0,2,2,SATURDAY,13,0,0,0,0,0,0,Hotel,None,0.698195,0.372334,0.1113,0.0573,0.9871,None,None,0.12,0.1034,0.3333,None,0.0505,None,0.1127,None,0.0,0.1134,0.0594,0.9871,None,None,0.1208,0.1034,0.3333,None,0.0516,None,0.1175,None,0.0,0.1124,0.0573,0.9871,None,None,0.12,0.1034,0.3333,None,0.0513,None,0.1148,None,0.0,None,block of flats,0.103,Panel,No,4.0,0.0,4.0,0.0,-1382.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,12.0,0.0,1.0
1,255041,0.0,Cash loans,F,N,Y,1,90000.0,298512.0,19206.0,270000.0,Family,Working,Secondary / secondary special,Married,House / apartment,0.019101,-16650,-909,-5176.0,-195,NaN,1,1,0,1,0,0,Cleaning staff,3.0,2,2,FRIDAY,15,0,0,0,0,1,1,Housing,None,0.565575,0.773896,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,4.0,0.0,4.0,0.0,-841.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,255042,1.0,Cash loans,F,Y,Y,2,112500.0,545040.0,26640.0,450000.0,Family,Working,Secondary / secondary special,Married,House / apartment,0.006207,-11091,-1349,-8324.0,-672,25.0,1,1,0,1,0,0,Sales staff,4.0,2,2,THURSDAY,13,0,0,0,0,0,0,Self-employed,None,0.004458,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,NaN,None,NaN,None,NaN,None,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
sql_distinct_credit_active = """
    SELECT distinct credit_active
    FROM bureau;
"""

credit_active = query_to_pd(sql_distinct_credit_active)
display(credit_active)

,credit_active
0,Active
1,Bad debt
2,Closed
3,Sold


In [8]:
statuses = credit_active["credit_active"].tolist()

sql_example = """
    SELECT b.credit_active, b.sk_id_bureau, bb.status
    FROM bureau b
    JOIN bureau_balance bb ON bb.sk_id_bureau = b.sk_id_bureau
    WHERE b.credit_active = %s
    LIMIT 1;
"""

with get_connection() as conn:
    for status in statuses:
        df = query_to_pd(sql_example, (status,), conn=conn)
        print(f"Статус: {status}")
        display(df)
        print()

Статус: Active


,credit_active,sk_id_bureau,status
0,Active,6241119,X



Статус: Bad debt


,credit_active,sk_id_bureau,status
0,Bad debt,5499851,X



Статус: Closed


,credit_active,sk_id_bureau,status
0,Closed,6374872,C



Статус: Sold


,credit_active,sk_id_bureau,status
0,Sold,5509002,X


In [ ]:
sql_distinct_status = """
    SELECT distinct status
    FROM bureau_balance;
"""

status = query_to_pd(sql_distinct_status)
display(status)

,status
0,0
1,1
2,2
3,3
4,4
5,5
6,C
7,X


In [10]:
sql_distinct_name_contract_status = """
    SELECT distinct name_contract_status
    FROM pos_cash_balance;
"""

name_contract_status = query_to_pd(sql_distinct_name_contract_status)
display(name_contract_status)

,name_contract_status
0,Active
1,Amortized debt
2,Approved
3,Canceled
4,Completed
5,Demand
6,Returned to the store
7,Signed
8,XNA


In [3]:
distinct_gender = """
    SELECT distinct code_gender
    FROM application_train_test
"""

gender = query_to_pd(distinct_gender)
display(gender)

,code_gender
0,F
1,M
2,XNA


In [11]:
sql_distinct_flag_document_18 = """
    SELECT distinct flag_document_18
    FROM application_train_test;
"""

flag_document_18 = query_to_pd(sql_distinct_flag_document_18)
display(flag_document_18)

,flag_document_18
0,0
1,1


## Выводы

Для колонок, которые представляют собой **id** в виде _числа_ наиболее подходящим типом является `integer`, так как он хорошо держит баланс между потреблением памяти и диапазоном значений. Тогда как `smallint` имеет диапазон *-32768 to +32767*, что недостаточно для типичного примера `sk_id_curr`, `sk_id_bureau`. При этом `bigint`, содержащий больший диапазон значений, требует в два раза больше памяти. Но по примерам значений id мы видим, что мы не приближаемся к правой границе диапазона `integer` и нет смысла использовать более широкий числовой тип.

Для колонок, которые предположительно являются словарями и имеют ограниченное количество значений, я бы выбирала между двумя типами `enum` или `character varying(n)`. Первый вариант идеально подходит для дней недели, например, колонки **weekday_appr_process_start**, потому что у нас точно не прибавится новое значение во время работы, но при этом день недели хорошо кодируется и числами от 1 до 7, что подходит для `smallint`, который занимает 2 байта вместо 4 байтов `enum`. Универсальный подход задавать эти колонки как `character varying(n)`: легко можно добавить новые значения, но формально, даже короткие строки занимают больше памяти, чем фиксированный `enum`. Учитывая padding в postgresql затраты по памяти могут не сильно различаться.
Если необходимо строго не допустить невалидных значений в колонке, то лучше `enum`, если такой необходимости нет, то лучше взять `character varying(n)`, где n чуть больше чем самое длинное значение из текущего словаря.

Для колонок, которые являются логическом типом или могут быть к нему приведены в целом можно использовать как и `char(1)`, `boolean`, `smallint`. Если так удобнее читать значения из колонок, например, **flag_own_car**: Y\N. При этом единого подхода хотелось бы придерживаться на уровне таблицы, а не, как, например, *application_train_test*, где встречаются и числовые и строковые колонки, имеющие бинарные значения, что вызывает трудности в выполнении различного анализа, когда `flag_own_car` это Y\N, но `flag_phone` это 0\1. Приходится постоянно смотреть на образец данных из таблицы.
Я бы предочла `boolean`, не только из-за того, что он занимает меньше по памяти, но и из-за того, что он не допускает сохранения невалидных значений. Если необходима большая гибкость в ответах Y\N\Unknown, то можно использовать `smallint` с кодировкой 1\0\\-1. Строковый тип я бы выбирала в последнюю очередь и только с точки зрения удобочитаемости данных.

Есть несколько колонок, которые относятся к денежным: колонки которые имеют в названии `amt`. Здесь используется тип *real*, но для денежных признаков это может быть небезопасно из-за ошибок округления. Я бы использовала `numeric`, имеющий большую точность. Однако, расчёты с этим типом могут быть заметно медленнее, чем с `integer` и занимает он 10+ байт памяти. Тип `real` позволяет быстро проводить математические расчёты, но имеет ошибки округления двоичных чисел с плавающей запятой и снижает точность при агрегировании. Для всех остальных числовых колонок, которые не являются денежными, я бы выбрала `real` для дробных и `integer` для целых чисел.

# Упражнения

## Вывести средний доход среди всех клиентов

In [11]:
sql_avg_income = """
    SELECT
        ROUND(AVG(amt_income_total)::numeric, 2) AS average_income
    FROM application_train_test;
"""

avg_income = query_to_pd(sql_avg_income)
display(avg_income)

,average_income
0,170116.06


## Вывести минимальный и максимальный возраст среди всех клиентов

In [4]:
sql_min_max_date = """
    SELECT
        MAX(FLOOR(-days_birth / 365.25)) as max_age,
        MIN(FLOOR(-days_birth / 365.25)) as min_age
    FROM application_train_test;
"""

min_max_age_table = query_to_pd(sql_min_max_date)
display(min_max_age_table)

,max_age,min_age
0,69,20


### Вывести количество мужчин и женщин

In [5]:
men_and_women = """
    SELECT
        COUNT(*) FILTER (WHERE code_gender = 'F') AS women,
        COUNT(*) FILTER (WHERE code_gender = 'M') AS men
    FROM application_train_test;
"""

gender_table = query_to_pd(men_and_women)
display(gender_table)

,women,men
0,235126,121125


## Вывести общую сумму, количество и среднюю сумму, запрошенную клиентами в кредит с авто и без

In [7]:
sql_creadit_car = """
    SELECT
        flag_own_car,
        SUM(amt_credit) AS total_sum,
        COUNT(*)        AS cnt,
        ROUND(AVG(amt_credit)::numeric, 2) AS avg_sum
    FROM application_train_test
    WHERE flag_own_car IN ('Y', 'N')
    GROUP BY flag_own_car
    ORDER BY flag_own_car;
"""

credit_car = query_to_pd(sql_creadit_car)
with option_context("display.float_format", "{:,.2f}".format):
    display(credit_car)

,flag_own_car,total_sum,cnt,avg_sum
0,N,"130,403,820,000.00",235235,554317.38
1,Y,"79,003,090,000.00",121020,652786.57


## Вывести доли клиентов с различным образованием

In [7]:
sql_precentage_eduaction = """
    SELECT
        name_education_type,
        COUNT(*) AS row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
    FROM application_train_test
    GROUP BY name_education_type
    ORDER BY percentage DESC;
"""

education_percentage = query_to_pd(sql_precentage_eduaction)
display(education_percentage)

,name_education_type,row_count,percentage
0,Secondary / secondary special,252379,70.84
1,Higher education,87379,24.53
2,Incomplete higher,12001,3.37
3,Lower secondary,4291,1.20
4,Academic degree,205,0.06


## Подсчитать количество полных лет для клиентов, у которых есть во владении автомобиль и недвижимость, вывести топ 10 по возрастанию

In [5]:
sql_max_age_realty_car = """
    SELECT age
    FROM (
        SELECT
            FLOOR(-days_birth / 365.25) AS age
        FROM application_train_test
        WHERE flag_own_car = 'Y'
        AND flag_own_realty = 'Y'
        ORDER BY age DESC
        LIMIT 10
    ) AS top10
    ORDER BY age ASC;
"""

max_age_realty_car_table = query_to_pd(sql_max_age_realty_car)
display(max_age_realty_car_table)

,age
0,68
1,68
2,68
3,68
4,68
5,68
6,68
7,68
8,68
9,68


## Вывести тех клиентов, у кого доход на одного члена семьи в два раза больше, чем в среднем на одного члена семьи по выборке

In [16]:
sql_above_avg_fam_income = """
    SELECT *
    FROM (
        SELECT
            sk_id_curr,
            amt_income_total,
            cnt_fam_members,
            ROUND((amt_income_total / cnt_fam_members)::numeric, 2) AS income_per_member,
            ROUND((AVG(amt_income_total / cnt_fam_members)
                OVER ())::numeric, 2) AS avg_income_per_member
        FROM application_train_test
        WHERE cnt_fam_members > 0
        AND amt_income_total IS NOT NULL
    ) t
    WHERE income_per_member > 2 * avg_income_per_member;
"""

above_avg_fam_income_table = query_to_pd(sql_above_avg_fam_income)
above_avg_fam_income_table["cnt_fam_members"] = above_avg_fam_income_table[
    "cnt_fam_members"
].astype(int)
above_avg_fam_income_table["income_per_member"] = above_avg_fam_income_table[
    "income_per_member"
].astype(float)
above_avg_fam_income_table["avg_income_per_member"] = (
    above_avg_fam_income_table["avg_income_per_member"].astype(float)
)
with option_context("display.float_format", "{:,.2f}".format):
    display(above_avg_fam_income_table)

,sk_id_curr,amt_income_total,cnt_fam_members,income_per_member,avg_income_per_member
0,394253,"202,500.00",1,"202,500.00","93,786.30"
1,394257,"216,000.00",1,"216,000.00","93,786.30"
2,394268,"202,500.00",1,"202,500.00","93,786.30"
3,394292,"207,000.00",1,"207,000.00","93,786.30"
4,394302,"382,500.00",2,"191,250.00","93,786.30"
...,...,...,...,...,...
27728,394089,"270,000.00",1,"270,000.00","93,786.30"
27729,394137,"202,500.00",1,"202,500.00","93,786.30"
27730,394160,"202,500.00",1,"202,500.00","93,786.30"
27731,394176,"450,000.00",2,"225,000.00","93,786.30"


# Вывести клиентов старше 60 лет по которым нет данных в bureau

In [6]:
sql_aged_clients_missed_in_bureau = """
    SELECT att.*
    FROM application_train_test att
    WHERE FLOOR(-days_birth / 365.25) >= 60
    AND NOT EXISTS (
      SELECT 1 FROM bureau b WHERE b.sk_id_curr = att.sk_id_curr
  );
"""

aged_missing_clients_table = query_to_pd(sql_aged_clients_missed_in_bureau)
display(aged_missing_clients_table)

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_18,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,249376,0.0,Cash loans,F,N,Y,0,135000.0,2517300.0,66402.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,106076,0.0,Cash loans,F,N,Y,0,31500.0,101880.0,10053.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,124764,0.0,Revolving loans,F,N,N,0,202500.0,360000.0,18000.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,446893,0.0,Cash loans,F,N,Y,0,74250.0,247275.0,19953.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,437448,0.0,Cash loans,F,N,Y,0,135000.0,781920.0,36364.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5928,320009,0.0,Cash loans,F,N,Y,0,112500.0,441481.5,25474.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5929,376243,0.0,Cash loans,F,N,Y,0,126000.0,714915.0,31621.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5930,369504,NaN,Cash loans,F,N,N,0,90000.0,114430.5,11853.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5931,352484,0.0,Cash loans,F,N,Y,0,157500.0,331920.0,15966.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


## Вывести женщин, у которых в истории bureau было больше двух кредитов, просроченных на 61 день и более, отсортировать в порядке убывания по кол-ву таких кредитов

In [27]:
sql_women_credit_overdue = """
    SELECT att.*, t.overdue_count
    FROM application_train_test att
    JOIN (
        SELECT
            b.sk_id_curr,
            COUNT(*) AS overdue_count
        FROM bureau b
        JOIN application_train_test att ON b.sk_id_curr = att.sk_id_curr
        WHERE b.credit_day_overdue >= 61
        AND att.code_gender = 'F'
        GROUP BY b.sk_id_curr
        HAVING COUNT(*) > 2
    ) t ON t.sk_id_curr = att.sk_id_curr
    ORDER BY overdue_count desc;
"""

women_credit_overdue_table = query_to_pd(sql_women_credit_overdue)
display(women_credit_overdue_table)

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,flag_document_19,flag_document_20,flag_document_21,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year,overdue_count
0,264144,1.0,Cash loans,F,N,Y,0,90000.0,254700.0,16582.5,...,0,0,0,0.0,0.0,0.0,1.0,0.0,1.0,5
1,142384,0.0,Cash loans,F,N,Y,0,180000.0,477621.0,37863.0,...,0,0,0,2.0,0.0,0.0,0.0,0.0,2.0,4
2,374345,0.0,Cash loans,F,N,Y,0,157500.0,301464.0,23949.0,...,0,0,0,1.0,0.0,0.0,0.0,0.0,3.0,3
3,375724,0.0,Cash loans,F,Y,N,0,45000.0,270000.0,12586.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,3
4,431820,0.0,Cash loans,F,N,Y,1,63000.0,104256.0,8194.5,...,0,0,0,1.0,0.0,0.0,0.0,1.0,1.0,3
5,114166,1.0,Cash loans,F,N,N,2,135000.0,640080.0,29970.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,3
6,436084,1.0,Cash loans,F,Y,Y,1,157500.0,269550.0,18364.5,...,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0,3
7,337741,1.0,Cash loans,F,N,N,0,108000.0,225000.0,17775.0,...,0,0,0,0.0,0.0,0.0,0.0,1.0,2.0,3


## По данным из bureau (БКИ) рассчитать долю просрочки в активных займах для каждого клиента, вывести топ 7 мужчин с наибольшей суммой просрочки, указав для них, помимо прочего, сумму активных кредитов и суммы всех кредитов (активных и закрытых).

In [8]:
sql_overdue_current_man = """
    SELECT
        b.sk_id_curr,
        ROUND(
            (
                SUM(b.amt_credit_sum_overdue)
                FILTER (WHERE b.credit_active = 'Active')
                /
                NULLIF(
                    SUM(b.amt_credit_sum) FILTER (WHERE b.credit_active = 'Active'),
                    0
                ))::numeric, 2) as overdue_ratio,
        SUM(b.amt_credit_sum_overdue)
            FILTER (WHERE b.credit_active = 'Active') AS total_overdue,
        SUM(b.amt_credit_sum)
            FILTER (WHERE b.credit_active = 'Active') as total_active,
        SUM(b.amt_credit_sum)
            FILTER (WHERE b.credit_active IN ('Active', 'Closed')) AS total_credits
    FROM bureau b
    JOIN application_train_test att ON att.sk_id_curr = b.sk_id_curr
    WHERE att.code_gender = 'M'
    GROUP BY b.sk_id_curr
    HAVING COUNT(*) FILTER (WHERE b.credit_active = 'Active') > 0
    ORDER BY total_overdue DESC
    LIMIT 7
"""

man_credit_overdue_table = query_to_pd(sql_overdue_current_man)
with option_context("display.float_format", "{:,.2f}".format):
    display(man_credit_overdue_table)

,sk_id_curr,overdue_ratio,total_overdue,total_active,total_credits
0,435405,1.00,"3,681,063.00","3,690,000.00","4,054,000.50"
1,427996,0.37,"1,571,697.00","4,231,615.50","5,631,583.50"
2,394113,0.87,"1,332,472.50","1,530,000.00","1,825,801.00"
3,266765,0.91,"1,224,474.90","1,350,000.00","1,421,955.00"
4,167085,4.93,"780,192.00","158,404.50","176,404.50"
5,154595,0.75,"742,491.00","990,000.00","4,245,754.50"
6,262411,0.54,"709,669.25","1,318,500.00","1,773,000.00"


# Новые признаки

## Показатель активности по БКИ

Сколько у клиентов записей в БКИ. Оценка верхней границы таблицы активности. Оконные фукнции используются на случай, если у двух клиентов одинаковое количество записей в БКИ.

In [5]:
sql_credit_activity = """
    SELECT
        sk_id_curr,
        COUNT(*) AS activity,
        RANK() OVER (ORDER BY COUNT(*) DESC) as rnk
    FROM bureau
    GROUP BY sk_id_curr
    ORDER BY rnk
    LIMIT 10
"""

credit_activity = query_to_pd(sql_credit_activity)
display(credit_activity)

,sk_id_curr,activity,rnk
0,120860,116,1
1,169704,94,2
2,318065,78,3
3,251643,61,4
4,425396,60,5
5,295809,59,6
6,129843,58,7
7,385133,57,8
8,177014,56,9
9,218175,55,10


## Категориальный признак `children_group` на основе количества детей:
- `Без детей` — 0 детей;
- `С детьми` — 1–2 ребёнка;
- `Многодетные` — 3 и более детей.

In [10]:
sql_children_group = """
    SELECT
        sk_id_curr,
        cnt_children,
        CASE
            WHEN cnt_children = 0 THEN 'Без детей'
            WHEN cnt_children IN (1, 2) THEN 'С детьми'
            ELSE 'Многодетные'
        END AS children_group
    FROM application_train_test;
"""

children_group = query_to_pd(sql_children_group)
display(children_group)

,sk_id_curr,cnt_children,children_group
0,299833,0,Без детей
1,299834,0,Без детей
2,299835,0,Без детей
3,299836,1,С детьми
4,299837,0,Без детей
...,...,...,...
356250,299828,1,С детьми
356251,299829,2,С детьми
356252,299830,1,С детьми
356253,299831,1,С детьми


## Категориальный признак `housing_group` на основе типа жилья клиента:
- `Собственное жильё` — дом или квартира;
- `Несобственное жильё` — арендованная или муниципальная квартира;
- `Другое` — остальные типы жилья.

In [11]:
sql_housing_group = """
    SELECT
        sk_id_curr,
        name_housing_type,
        CASE
            WHEN name_housing_type = 'House / apartment'
                THEN 'Собственное жильё'
            WHEN name_housing_type IN (
                'Rented apartment',
                'Municipal apartment'
            )
                THEN 'Несобственное жильё'
            ELSE 'Другое'
        END AS housing_group
    FROM application_train_test;
"""

housing_group = query_to_pd(sql_housing_group)
display(housing_group)

,sk_id_curr,name_housing_type,housing_group
0,299833,House / apartment,Собственное жильё
1,299834,House / apartment,Собственное жильё
2,299835,Rented apartment,Несобственное жильё
3,299836,House / apartment,Собственное жильё
4,299837,House / apartment,Собственное жильё
...,...,...,...
356250,299828,House / apartment,Собственное жильё
356251,299829,House / apartment,Собственное жильё
356252,299830,With parents,Другое
356253,299831,House / apartment,Собственное жильё


## Средняя сумма активного долга клиента

In [22]:
sql_avg_active_debt = """
    SELECT
        sk_id_curr,
        ROUND(
            SUM(amt_credit_sum_debt)::numeric
            / COUNT(*),
            2
        ) AS avg_active_debt
    FROM bureau
    WHERE credit_active = 'Active'
      AND amt_credit_sum_debt IS NOT NULL
      AND amt_credit_sum_debt >= 0
    GROUP BY sk_id_curr
    ORDER BY avg_active_debt DESC;
"""

avg_active_debt = query_to_pd(sql_avg_active_debt)
avg_active_debt["avg_active_debt"] = (
    avg_active_debt["avg_active_debt"].astype(float)
)
with option_context("display.float_format", "{:,.2f}".format):
    display(avg_active_debt)

,sk_id_curr,avg_active_debt
0,363076,"83,624,500.00"
1,446029,"62,219,000.00"
2,161114,"51,365,200.00"
3,308724,"43,650,000.00"
4,355169,"43,094,400.00"
...,...,...
239364,395202,0.00
239365,449040,0.00
239366,442718,0.00
239367,453968,0.00


## Превышение текущей задолженности над первоначальной суммой займа

In [17]:
sql_debt_exceeds_credit = """
    SELECT
        sk_id_curr,
        SUM(amt_credit_sum_debt) AS active_debt,
        SUM(amt_credit_sum) AS active_credit_sum,
        SUM(amt_credit_sum_debt) / SUM(amt_credit_sum) AS ratio
    FROM bureau
    WHERE credit_active = 'Active'
    AND amt_credit_sum > 0
    AND amt_credit_sum_debt >= 0
    GROUP BY sk_id_curr
    HAVING SUM(amt_credit_sum_debt) > SUM(amt_credit_sum);
"""

debt_exceeds_credit = query_to_pd(sql_debt_exceeds_credit)
display(debt_exceeds_credit)

,sk_id_curr,active_debt,active_credit_sum,ratio
0,157514,194931.94,45000.0,4.331821
1,315990,6959178.00,6642000.0,1.047753
2,327330,205335.00,202500.0,1.014000
3,244185,3485367.00,2603497.5,1.338725
4,347279,49680.00,45000.0,1.104000
...,...,...,...,...
6709,276754,2054360.20,1838817.0,1.117218
6710,128268,541062.00,540000.0,1.001967
6711,199855,1139625.00,1125000.0,1.013000
6712,447678,1625919.40,1620000.0,1.003654
